In [ ]:
from dataset import ChessDataset
from model import ChessCNN
from encoder import board_to_tensor
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import math
import time

BATCH_SIZE = 250000
TOTAL_SIZE = 12900000

NUM_BATCH = math.ceil(TOTAL_SIZE/BATCH_SIZE)

In [ ]:
X_chunks = []
Y_chunks = []

for chunk_id in range(52):
    X_chunks.append(torch.load(fr"D:\Coding\CNN Chess Engine\data\pre-tensors\X_{chunk_id+1}.pt"))
    Y_chunks.append(torch.load(fr"D:\Coding\CNN Chess Engine\data\pre-tensors\Y_{chunk_id+1}.pt"))

X = torch.cat(X_chunks,dim=0)
Y = torch.cat(Y_chunks,dim=0)


In [ ]:
from dataset import ChessDataset
from model import ChessCNN
from encoder import board_to_tensor
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import math
import time

BATCH_SIZE = 250000
TOTAL_SIZE = 12900000

NUM_BATCH = math.ceil(TOTAL_SIZE/BATCH_SIZE)

In [ ]:
from dataset import ChessDataset
from model import ChessCNN
from encoder import board_to_tensor
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import math
import time

BATCH_SIZE = 250000
TOTAL_SIZE = 12900000

NUM_BATCH = math.ceil(TOTAL_SIZE/BATCH_SIZE)

In [ ]:
import sys
import torch

print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

In [ ]:
device = torch.device("cuda")

model = ChessCNN().to(device)
model.load_state_dict(
    torch.load(
        fr"d:\Coding\CNN Chess Engine\data\chess_models\best_model.pth",
        map_location=device
    )

)

In [ ]:
dataset = TensorDataset(X, Y)

train_size = int(0.98*len(dataset))
val_size = len(dataset) - train_size

train_dataset , val_dataset = random_split(dataset,[train_size,val_size])

In [ ]:
def train_one_epoch(model,loader,optimizer,criterion,device):
    
    model.train()

    running_loss = 0.0
    
    for boards,targets in loader:
        boards = boards.float().to(device)
        targets = targets.to(device)

        predictions = model(boards)

        loss = criterion(predictions,targets.unsqueeze(1))
        
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()
        
        running_loss+=loss.item()

    avg_loss = running_loss/len(loader)
    
    return avg_loss

def evaluate(model,loader,criterion,device):
    model.eval()

    running_loss =0

    with torch.no_grad():

        for boards,targets in loader:
            boards = boards.float().to(device)
            targets = targets.to(device)

            predictions = model(boards)

            loss = criterion(predictions,targets.unsqueeze(1))

            running_loss +=loss.item()
    
    avg_loss = running_loss/len(loader)

    return avg_loss

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True,num_workers=2,)
test_loader = DataLoader(val_dataset,batch_size=64,shuffle=True,num_workers=2,)

optimizer = optim.Adam(model.parameters(),lr=0.001)

criterion = nn.MSELoss()

best_val_loss = float("inf")

start_time = time.time()

num_epochs = 1
best_epoch =-1

for epoch in range(num_epochs):

    train_loss = train_one_epoch(model,train_loader,optimizer,criterion,device)

    val_loss = evaluate(model,test_loader,criterion,device)

    torch.save(model.state_dict(),fr"d:\Coding\CNN Chess Engine\data\chess_models\epoch_{epoch+1}.pth")

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      best_epoch = epoch+1
      torch.save(model.state_dict(),fr"d:\Coding\CNN Chess Engine\data\chess_models\best_model.pth")
      print(f"Saved Best Model: {epoch+1}/{num_epochs}")



    print(f"Epoch: {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")

total_time = time.time() - start_time
print(f"\nTotal Training Time: {total_time/60:.2f} minutes")
print(f"Average Epoch Time: {total_time/num_epochs:.2f} seconds")


In [ ]:
for i in range(20):

    board, target = test_dataset[i]

    board = board.unsqueeze(0).to(device)

    prediction = model(board)

    print("Target:", target.item())
    print("Prediction:", prediction.item())